In [17]:
# -----------------------------
# 0. Mount Drive + paths
# -----------------------------
from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# Phase 3: RFantibody PDBs -> VH/VL extraction -> ESM2 scoring
#
# Input:
#   Extracted RFantibody PDBs with chains:
#       H = VH
#       L = VL
#       T = HER2 target
#
# Output:
#   Reranked candidate CSVs with ESM2 binding/expression scores
# ============================================================

!pip install -q transformers huggingface_hub pandas tqdm biopython

from pathlib import Path
import re

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from transformers import AutoTokenizer, EsmModel
from huggingface_hub import hf_hub_download


# ============================================================
# 0. Paths / config
# ============================================================

PDB_DIR = Path("/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted")

OUT_DIR = Path("/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/esm2_scores")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATE_CSV = OUT_DIR / "her2_rfantibody_vh_vl_from_pdb.csv"
RERANKED_CSV = OUT_DIR / "her2_rfantibody_phase3_esm2_reranked_candidates.csv"
TOP10_CSV = OUT_DIR / "her2_rfantibody_phase3_top10_candidates.csv"

HF_REPO = "zhangtin/antibody-esm2-finetuned"

# IMPORTANT: use the same base model as your fine-tuning checkpoint.
BASE_MODEL = "facebook/esm2_t33_650M_UR50D"

METRICS = {
    "binding_koenig": "koenig_binding_g6/fold0_best.pt",
    "expression_koenig": "koenig_expression_g6/fold0_best.pt",
    "binding_warszawski": "warszawski_binding_d44/fold0_best.pt",
}

AA = set("ACDEFGHIKLMNPQRSTVWY")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("PDB_DIR:", PDB_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
PDB_DIR: /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted


In [9]:
# ============================================================
# 1. Extract VH/VL/T sequences from PDB files
# ============================================================

def clean_seq(seq: str) -> str:
    seq = str(seq).upper()
    seq = re.sub(r"[^A-Z]", "", seq)
    return seq


def valid_aa_seq(seq: str, min_len=30, max_len=1022) -> bool:
    return min_len <= len(seq) <= max_len and set(seq).issubset(AA)


def pdb_chain_sequences(pdb_path: Path) -> dict:
    """
    Read one PDB and return {chain_id: amino_acid_sequence}.
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_path.stem, pdb_path)
    model = list(structure.get_models())[0]

    chains = {}

    for chain in model:
        seq = []
        for residue in chain:
            if residue.id[0] != " ":
                continue

            try:
                seq.append(seq1(residue.resname))
            except Exception:
                seq.append("X")

        chains[chain.id] = clean_seq("".join(seq))

    return chains


pdb_files = sorted(PDB_DIR.glob("*.pdb"))

print("Found PDB files:", len(pdb_files))
print(pdb_files[:5])

Found PDB files: 56
[PosixPath('/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted/samples_design_0_dldesign_0.pdb'), PosixPath('/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted/samples_design_0_dldesign_1.pdb'), PosixPath('/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted/samples_design_0_dldesign_2.pdb'), PosixPath('/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted/samples_design_0_dldesign_3.pdb'), PosixPath('/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/extracted/samples_design_0_dldesign_4.pdb')]


In [10]:
# ============================================================
# 2. Build candidate table
#
# Expected RFantibody chains:
#   H = heavy variable chain / VH
#   L = light variable chain / VL
#   T = HER2 target
# ============================================================

rows = []

for pdb_path in pdb_files:
    chains = pdb_chain_sequences(pdb_path)

    vh = chains.get("H", "")
    vl = chains.get("L", "")
    target_seq = chains.get("T", "")

    # Skip non-antibody / broken structures
    if not valid_aa_seq(vh, min_len=80, max_len=150):
        continue
    if not valid_aa_seq(vl, min_len=80, max_len=150):
        continue

    rows.append({
        "candidate_id": pdb_path.stem,
        "target": "HER2",
        "run_id": "her2_rfantibody",
        "sequence_generator": "RFantibody_ProteinMPNN",
        "source_pdb": str(pdb_path),
        "vh": vh,
        "vl": vl,
        "target_sequence": target_seq,
        "vh_length": len(vh),
        "vl_length": len(vl),
        "target_length": len(target_seq),
    })

df = pd.DataFrame(rows)

print("Usable VH/VL candidates:", len(df))
display(df[["candidate_id", "vh_length", "vl_length", "target_length"]].head())

df.to_csv(CANDIDATE_CSV, index=False)
print("Saved:", CANDIDATE_CSV)

Usable VH/VL candidates: 56


,candidate_id,vh_length,vl_length,target_length
0,samples_design_0_dldesign_0,114,107,214
1,samples_design_0_dldesign_1,114,107,214
2,samples_design_0_dldesign_2,114,107,214
3,samples_design_0_dldesign_3,114,107,214
4,samples_design_0_dldesign_4,114,107,214


Saved: /content/drive/MyDrive/her2_project/rfantibody_her2/outputs/esm2_scores/her2_rfantibody_vh_vl_from_pdb.csv


In [11]:
# ============================================================
# 3. Define fine-tuned ESM2 model
#
# This matches your training script:
#   ESM2(VH) -> mean pool
#   ESM2(VL) -> mean pool
#   concat(VH_embedding, VL_embedding)
#   regression head
#
# Your appended file documents this VH/VL training setup.
# ============================================================

class ESM2RegressionHead(nn.Module):
    def __init__(self, hidden_size: int, dropout: float = 0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(hidden_size * 2),
            nn.Linear(hidden_size * 2, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class FineTunedESM2(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()

        self.esm2 = EsmModel.from_pretrained(model_name)
        self.head = ESM2RegressionHead(self.esm2.config.hidden_size)

    @staticmethod
    def mean_pool(last_hidden_state, attention_mask, special_tokens_mask):
        valid = attention_mask.bool() & ~special_tokens_mask.bool()
        valid = valid.unsqueeze(-1).float()

        summed = (last_hidden_state * valid).sum(dim=1)
        counts = valid.sum(dim=1).clamp(min=1)

        return summed / counts

    def encode(self, enc):
        out = self.esm2(
            input_ids=enc["input_ids"],
            attention_mask=enc["attention_mask"],
        )

        return self.mean_pool(
            out.last_hidden_state,
            enc["attention_mask"],
            enc["special_tokens_mask"],
        )

    def forward(self, heavy_enc, light_enc):
        h = self.encode(heavy_enc)
        l = self.encode(light_enc)

        return self.head(torch.cat([h, l], dim=-1))

In [12]:
# ============================================================
# 4. Load one fine-tuned checkpoint
# ============================================================

def load_metric_model(metric_name: str, ckpt_file: str):
    ckpt_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=ckpt_file,
        repo_type="model",
    )

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ckpt["state_dict"] if isinstance(ckpt, dict) and "state_dict" in ckpt else ckpt

    model = FineTunedESM2(BASE_MODEL)
    missing, unexpected = model.load_state_dict(state, strict=False)

    model.to(device)
    model.eval()

    print(f"\nLoaded metric: {metric_name}")
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))

    if isinstance(ckpt, dict) and "val_metrics" in ckpt:
        keep = ["spearman", "pearson", "rmse", "mae", "r2"]
        print("Validation:", {k: ckpt["val_metrics"].get(k) for k in keep if k in ckpt["val_metrics"]})

    return model

In [13]:
# ============================================================
# 5. Score VH/VL pairs
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

@torch.no_grad()
def score_vh_vl(model, vh_list, vl_list, batch_size=4):
    scores = []

    for i in tqdm(range(0, len(vh_list), batch_size)):
        vh_batch = vh_list[i:i + batch_size]
        vl_batch = vl_list[i:i + batch_size]

        heavy_enc = tokenizer(
            vh_batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_special_tokens_mask=True,
            return_tensors="pt",
        ).to(device)

        light_enc = tokenizer(
            vl_batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_special_tokens_mask=True,
            return_tensors="pt",
        ).to(device)

        preds = model(heavy_enc, light_enc)

        scores.extend(
            preds.detach()
            .cpu()
            .float()
            .numpy()
            .tolist()
        )

    return np.array(scores, dtype=float)


vh_list = df["vh"].tolist()
vl_list = df["vl"].tolist()

for metric_name, ckpt_file in METRICS.items():
    model = load_metric_model(metric_name, ckpt_file)

    df[f"esm2_{metric_name}"] = score_vh_vl(
        model=model,
        vh_list=vh_list,
        vl_list=vl_list,
        batch_size=4,
    )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

score_cols = [c for c in df.columns if c.startswith("esm2_")]

display(df[["candidate_id"] + score_cols].head())

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Loaded metric: binding_warszawski
Missing keys: 33
Unexpected keys: 1
Validation: {'spearman': np.float64(0.6019287411889391), 'pearson': np.float32(0.59255093), 'rmse': 1.0540799687032516, 'mae': 0.8293677568435669, 'r2': 0.3351668119430542}


  0%|          | 0/14 [00:00<?, ?it/s]

,candidate_id,esm2_binding_koenig,esm2_expression_koenig,esm2_binding_warszawski
0,samples_design_0_dldesign_0,10.480033,0.066459,15.490098
1,samples_design_0_dldesign_1,10.390663,0.074953,14.925531
2,samples_design_0_dldesign_2,10.344128,0.093052,14.967990
3,samples_design_0_dldesign_3,10.432817,0.098863,15.008526
4,samples_design_0_dldesign_4,10.518681,0.090956,15.269972


In [19]:
# ============================================================
# 6. Composite ranking
#
# Higher ESM2 predicted score = better.
# Use z-scores so different metrics are comparable.
# ============================================================

def zscore(x):
    return (x - x.mean()) / (x.std() + 1e-8)


for c in score_cols:
    df[c + "_z"] = zscore(df[c])

zcols = [c for c in df.columns if c.endswith("_z")]

df["phase3_composite_score"] = df[zcols].mean(axis=1)

df = df.sort_values(
    "phase3_composite_score",
    ascending=False,
).reset_index(drop=True)

df["phase3_rank"] = np.arange(1, len(df) + 1)

display(
    df[
        [
            "phase3_rank",
            "candidate_id",
            "phase3_composite_score",
        ]
        + score_cols
        + ["vh_length", "vl_length", "source_pdb"]
    ].head(20)
)


# ============================================================
# 7. Save reranked outputs
# ============================================================

df.to_csv(RERANKED_CSV, index=False)
df.head(10).to_csv(TOP10_CSV, index=False)

print("Saved:")
print(RERANKED_CSV)
print(TOP10_CSV)

# ============================================================
# 8. Save compact Phase 4 handoff
#
# This is what you feed into Chai / AF-Multimer validation.
# ============================================================

phase4_cols = [
    "phase3_rank",
    "candidate_id",
    "target",
    "run_id",
    "sequence_generator",
    "source_pdb",
    "vh",
    "vl",
    "target_sequence",
    "vh_length",
    "vl_length",
    "target_length",
    "phase3_composite_score",
] + score_cols + zcols

phase4_cols = [c for c in phase4_cols if c in df.columns]

PHASE4_CSV = OUT_DIR / "her2_rfantibody_phase4_structure_validation_handoff.csv"

df.loc[df["phase3_rank"] <= 50, phase4_cols].to_csv(
    PHASE4_CSV,
    index=False,
)

print("Phase 4 handoff saved:")
print(PHASE4_CSV)

,phase3_rank,candidate_id,phase3_composite_score,esm2_binding_koenig,esm2_expression_koenig,esm2_binding_warszawski,vh_length,vl_length,source_pdb
0,1,samples_design_6_dldesign_2,1.651125,10.993450,0.229143,15.721763,118,104,/content/drive/MyDrive/her2_project/rfantibody...
1,2,samples_design_6_dldesign_3,1.634978,11.004610,0.172090,16.054016,118,104,/content/drive/MyDrive/her2_project/rfantibody...
2,3,samples_design_6_dldesign_4,1.227079,11.013252,0.219899,15.441420,118,104,/content/drive/MyDrive/her2_project/rfantibody...
3,4,samples_design_6_dldesign_7,1.141807,11.065998,0.139281,15.822550,118,104,/content/drive/MyDrive/her2_project/rfantibody...
4,5,samples_design_6_dldesign_5,0.876620,10.966637,0.137348,15.749667,118,104,/content/drive/MyDrive/her2_project/rfantibody...
5,6,samples_design_5_dldesign_5,0.809795,10.942856,0.118363,15.845599,122,104,/content/drive/MyDrive/her2_project/rfantibody...
6,7,samples_design_5_dldesign_6,0.789205,10.929867,0.125504,15.800298,122,104,/content/drive/MyDrive/her2_project/rfantibody...
7,8,samples_design_2_dldesign_2,0.786105,10.495969,0.227723,15.649741,115,108,/content/drive/MyDrive/her2_project/rfantibody...
8,9,samples_design_5_dldesign_7,0.730915,10.791503,0.142691,15.806019,122,104,/content/drive/MyDrive/her2_project/rfantibody...
9,10,samples_design_5_dldesign_1,0.720823,10.909433,0.103472,15.910344,122,104,/content/drive/MyDrive/her2_project/rfantibody...


Saved:
/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/esm2_scores/her2_rfantibody_phase3_esm2_reranked_candidates.csv
/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/esm2_scores/her2_rfantibody_phase3_top10_candidates.csv
Phase 4 handoff saved:
/content/drive/MyDrive/her2_project/rfantibody_her2/outputs/esm2_scores/her2_rfantibody_phase4_structure_validation_handoff.csv
